# 02 - Ollama API Basics

This notebook walks through direct Ollama inference against your k3s deployment and builds latency charts.

Recommended pre-step (separate terminal):
```bash
kubectl port-forward -n llm-observability svc/ollama 11434:11434
```


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
# If localhost:11434 is unavailable, run this in a separate terminal before continuing:
#!kubectl port-forward -n llm-observability svc/ollama 11434:11434


In [ ]:
import json
import time

import matplotlib.pyplot as plt
import pandas as pd
import requests

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_PORT_FORWARD_CMD = "kubectl port-forward -n llm-observability svc/ollama 11434:11434"
MODEL_NAME = "gemma3-1b-it-gguf-local"


def is_http_available(url: str, timeout: int = 3) -> bool:
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code < 500
    except Exception:
        return False


OLLAMA_AVAILABLE = is_http_available(f"{OLLAMA_BASE_URL}/api/tags")
print("OLLAMA_BASE_URL:", OLLAMA_BASE_URL)
print("MODEL_NAME      :", MODEL_NAME)
print("OLLAMA_AVAILABLE:", OLLAMA_AVAILABLE)
if not OLLAMA_AVAILABLE:
    print("Local Ollama endpoint is not reachable on localhost:11434.")
    print("In another terminal run:")
    print(f"  {OLLAMA_PORT_FORWARD_CMD}")


## Discover Installed Models


In [ ]:
models = []
payload = {"models": []}

if not OLLAMA_AVAILABLE:
    print("Skipping model discovery: Ollama endpoint is not reachable on localhost:11434")
else:
    try:
        resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=15)
        resp.raise_for_status()
        payload = resp.json()
        models = payload.get("models", [])
    except Exception as exc:
        print(f"Model discovery failed: {exc}")

print(f"Discovered {len(models)} model(s)")
for item in models:
    print("-", item.get("name"), "size:", item.get("size"))


## Single Non-Streaming Chat


In [ ]:
prompt = "Explain in 3 bullets why observability matters for local LLM stacks."
body = {
    "model": MODEL_NAME,
    "stream": False,
    "messages": [{"role": "user", "content": prompt}],
}

if not OLLAMA_AVAILABLE:
    print("Skipping chat test: Ollama endpoint is not reachable on localhost:11434")
else:
    try:
        start = time.perf_counter()
        resp = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=body, timeout=180)
        elapsed = (time.perf_counter() - start) * 1000
        resp.raise_for_status()
        chat = resp.json()
        print(f"HTTP {resp.status_code} in {elapsed:.2f} ms")
        print(chat.get("message", {}).get("content", "")[:1200])
    except Exception as exc:
        print(f"Chat test failed: {exc}")


## Streaming Chat


In [ ]:
stream_body = {
    "model": MODEL_NAME,
    "stream": True,
    "messages": [{"role": "user", "content": "Provide a short rollout checklist for a Kubernetes deployment."}],
}

if not OLLAMA_AVAILABLE:
    print("Skipping streaming test: Ollama endpoint is not reachable on localhost:11434")
else:
    try:
        with requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=stream_body, stream=True, timeout=180) as resp:
            resp.raise_for_status()
            chunks = []
            for line in resp.iter_lines(decode_unicode=True):
                if not line:
                    continue
                data = json.loads(line)
                if "message" in data and "content" in data["message"]:
                    chunks.append(data["message"]["content"])
                if data.get("done"):
                    break

        print("".join(chunks)[:1500])
    except Exception as exc:
        print(f"Streaming test failed: {exc}")


## Multi-Prompt Benchmark


In [ ]:
prompts = [
    "What is a readiness probe?",
    "How does LangSmith help debugging?",
    "When should I scale down local Kubernetes workloads?",
    "Give one GPU troubleshooting tip for Ollama.",
]

rows = []
if not OLLAMA_AVAILABLE:
    print("Skipping benchmark: Ollama endpoint is not reachable on localhost:11434")
else:
    for i in range(12):
        p = prompts[i % len(prompts)]
        body = {
            "model": MODEL_NAME,
            "stream": False,
            "messages": [{"role": "user", "content": p}],
        }
        start = time.perf_counter()
        try:
            r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=body, timeout=180)
            elapsed = (time.perf_counter() - start) * 1000
            status = r.status_code
            eval_count = None
            total_duration = None
            if status == 200:
                j = r.json()
                eval_count = j.get("eval_count")
                total_duration = j.get("total_duration")
            rows.append(
                {
                    "iteration": i + 1,
                    "status": status,
                    "latency_ms": round(elapsed, 2),
                    "eval_count": eval_count,
                    "total_duration_ns": total_duration,
                    "prompt": p,
                }
            )
        except Exception as exc:
            elapsed = (time.perf_counter() - start) * 1000
            rows.append(
                {
                    "iteration": i + 1,
                    "status": -1,
                    "latency_ms": round(elapsed, 2),
                    "eval_count": None,
                    "total_duration_ns": None,
                    "prompt": p,
                    "error": str(exc),
                }
            )

if rows:
    df = pd.DataFrame(rows)
else:
    df = pd.DataFrame(columns=["iteration", "status", "latency_ms", "eval_count", "total_duration_ns", "prompt", "error"])

df


In [ ]:
if not df.empty:
    print("Average latency (ms):", round(df["latency_ms"].mean(), 2))
    print("P95 latency (ms)    :", round(df["latency_ms"].quantile(0.95), 2))

    ax = df.plot(x="iteration", y="latency_ms", marker="o", figsize=(10, 4), title="Ollama Chat Latency")
    ax.set_ylabel("Latency (ms)")
    plt.show()


## Done
Next: `03-langchain-proxy-deep-dive.ipynb` to compare direct Ollama calls with your traced proxy path.
